# E-Commerce Transactions Analysis
**Dataset**: smayanj/e-commerce-transactions-dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('shardul/ecommerce-transactions/ecommerce_transactions.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()


## 1. Data Quality & Schema

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Null Counts ===')
print(df.isnull().sum())
print(f'\n=== Unique Values ===')
for col in df.columns:
    print(f'{col}: {df[col].nunique():,}')


## 2. Statistical Summary

In [ ]:
df.describe()


## 3. Categorical Distributions

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
n = len(cat_cols)
if n > 0:
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 5*((n+1)//2)))
    axes = axes.flatten() if n > 2 else [axes] if n == 1 else axes.flatten()
    for i, col in enumerate(cat_cols):
        if df[col].nunique() <= 30:
            df[col].value_counts().plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
            axes[i].set_title(f'{col} Distribution')
            axes[i].tick_params(axis='x', rotation=45)
        else:
            df[col].value_counts().head(15).plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'))
            axes[i].set_title(f'{col} (Top 15)')
            axes[i].tick_params(axis='x', rotation=45)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 4. Temporal Analysis

In [ ]:
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f'{col}: {df[col].min()} to {df[col].max()} ({(df[col].max()-df[col].min()).days} days)')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    df[col].dt.dayofweek.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title(f'{col} - Day of Week')
    axes[0].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
    
    df[col].dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title(f'{col} - Monthly Distribution')
    plt.tight_layout()
    plt.show()


## 5. Transaction Value Analysis

In [ ]:
price_cols = [c for c in df.columns if any(k in c.lower() for k in ['price','amount','total','revenue'])]
qty_cols = [c for c in df.columns if any(k in c.lower() for k in ['quantity','qty'])]
value_cols = price_cols + qty_cols

if value_cols:
    fig, axes = plt.subplots(1, len(value_cols), figsize=(7*len(value_cols), 5))
    if len(value_cols) == 1:
        axes = [axes]
    for i, col in enumerate(value_cols):
        if df[col].dtype in ['float64','int64']:
            axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
            axes[i].set_title(f'{col} Distribution')
            axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
            axes[i].legend()
    plt.tight_layout()
    plt.show()

for col in price_cols:
    if df[col].dtype in ['float64','int64']:
        print(f'{col} — Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f}, Total: {df[col].sum():,.2f}')


## 6. Customer Analysis

In [ ]:
cust_cols = [c for c in df.columns if any(k in c.lower() for k in ['customer','user','client'])]
if cust_cols:
    cc = cust_cols[0]
    print(f'Unique customers: {df[cc].nunique():,}')
    
    cust_txn = df.groupby(cc).size()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(cust_txn.values, bins=50, edgecolor='black', alpha=0.7, color='teal')
    ax.set_title('Transactions per Customer')
    ax.set_xlabel('Number of Transactions')
    ax.axvline(cust_txn.mean(), color='red', linestyle='--', label=f'Mean: {cust_txn.mean():.1f}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    if price_cols:
        cust_spend = df.groupby(cc)[price_cols[0]].sum()
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.hist(cust_spend.values, bins=50, edgecolor='black', alpha=0.7, color='salmon')
        ax.set_title(f'Total {price_cols[0]} per Customer')
        ax.axvline(cust_spend.mean(), color='red', linestyle='--', label=f'Mean: {cust_spend.mean():.2f}')
        ax.legend()
        plt.tight_layout()
        plt.show()


## 7. Product Analysis

In [ ]:
prod_cols = [c for c in df.columns if any(k in c.lower() for k in ['product','item','sku'])]
cat_cols_prod = [c for c in df.columns if 'category' in c.lower()]

if prod_cols:
    pc = prod_cols[0]
    print(f'Unique products: {df[pc].nunique():,}')
    
    top = df[pc].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(14, 8))
    top.plot(kind='barh', ax=ax, color=sns.color_palette('magma', 20))
    ax.set_title(f'Top 20 Products by Frequency')
    ax.set_xlabel('Count')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

if cat_cols_prod and price_cols:
    for col in cat_cols_prod:
        fig, ax = plt.subplots(figsize=(14, 6))
        cat_rev = df.groupby(col)[price_cols[0]].sum().sort_values(ascending=True)
        cat_rev.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(cat_rev)))
        ax.set_title(f'Revenue by {col}')
        ax.set_xlabel(f'Total {price_cols[0]}')
        plt.tight_layout()
        plt.show()


## 8. Relevance to Shelf Optimization / Planogram AI

**Strengths:**
- Revenue/price data enables margin-based SKU scoring
- Product velocity from transaction frequency
- Customer basket analysis for co-purchase detection
- Category performance for shelf allocation

**Limitations:**
- E-commerce data — no physical shelf/aisle information
- May not directly map to in-store shopping patterns
- No physical constraint data (shelf dimensions, facings)


---

# Purchase Pattern Analysis

The following sections analyze customer purchasing behavior, demographic preferences, geographic patterns, and cohort-based insights.

## 9. Customer Lifetime Value (CLV) Analysis

In [ ]:
# Ensure Transaction_Date is datetime
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Calculate CLV metrics per customer
clv_df = df.groupby('User_Name').agg(
    total_spend=('Purchase_Amount', 'sum'),
    transaction_count=('Transaction_ID', 'count'),
    avg_order_value=('Purchase_Amount', 'mean'),
    first_purchase=('Transaction_Date', 'min'),
    last_purchase=('Transaction_Date', 'max')
).reset_index()

# Calculate customer lifespan in days
clv_df['lifespan_days'] = (clv_df['last_purchase'] - clv_df['first_purchase']).dt.days

# Simple CLV calculation: Avg Order Value * Purchase Frequency (per month) * Avg Lifespan (months)
avg_lifespan_months = clv_df['lifespan_days'].mean() / 30
clv_df['purchases_per_month'] = clv_df['transaction_count'] / (clv_df['lifespan_days'] / 30).replace(0, 1)
clv_df['simple_clv'] = clv_df['avg_order_value'] * clv_df['purchases_per_month'] * 12  # Annualized CLV

print('=== Customer Lifetime Value Summary ===')
print(f"Total Customers: {len(clv_df):,}")
print(f"Avg Total Spend per Customer: ${clv_df['total_spend'].mean():,.2f}")
print(f"Avg Transaction Count per Customer: {clv_df['transaction_count'].mean():.1f}")
print(f"Avg Order Value: ${clv_df['avg_order_value'].mean():.2f}")
print(f"Avg Customer Lifespan: {clv_df['lifespan_days'].mean():.0f} days ({clv_df['lifespan_days'].mean()/30:.1f} months)")
print(f"Avg Annualized CLV: ${clv_df['simple_clv'].mean():,.2f}")

# Display top 10 customers by CLV
print('\n=== Top 10 Customers by Total Spend ===')
clv_df.nlargest(10, 'total_spend')[['User_Name', 'total_spend', 'transaction_count', 'avg_order_value', 'simple_clv']]

In [ ]:
# CLV Distribution Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Total Spend Distribution
axes[0, 0].hist(clv_df['total_spend'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].axvline(clv_df['total_spend'].mean(), color='red', linestyle='--', label=f"Mean: ${clv_df['total_spend'].mean():,.0f}")
axes[0, 0].axvline(clv_df['total_spend'].median(), color='orange', linestyle='--', label=f"Median: ${clv_df['total_spend'].median():,.0f}")
axes[0, 0].set_title('Total Spend per Customer Distribution')
axes[0, 0].set_xlabel('Total Spend ($)')
axes[0, 0].legend()

# Transaction Count Distribution
axes[0, 1].hist(clv_df['transaction_count'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].axvline(clv_df['transaction_count'].mean(), color='red', linestyle='--', label=f"Mean: {clv_df['transaction_count'].mean():.0f}")
axes[0, 1].set_title('Transaction Count per Customer Distribution')
axes[0, 1].set_xlabel('Number of Transactions')
axes[0, 1].legend()

# Average Order Value Distribution
axes[1, 0].hist(clv_df['avg_order_value'], bins=30, edgecolor='black', alpha=0.7, color='teal')
axes[1, 0].axvline(clv_df['avg_order_value'].mean(), color='red', linestyle='--', label=f"Mean: ${clv_df['avg_order_value'].mean():.0f}")
axes[1, 0].set_title('Average Order Value per Customer Distribution')
axes[1, 0].set_xlabel('Average Order Value ($)')
axes[1, 0].legend()

# CLV Distribution
axes[1, 1].hist(clv_df['simple_clv'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].axvline(clv_df['simple_clv'].mean(), color='red', linestyle='--', label=f"Mean: ${clv_df['simple_clv'].mean():,.0f}")
axes[1, 1].set_title('Annualized CLV Distribution')
axes[1, 1].set_xlabel('CLV ($)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Customer Segmentation by CLV
clv_df['clv_segment'] = pd.qcut(clv_df['simple_clv'], q=4, labels=['Low', 'Medium', 'High', 'Premium'])
print('\n=== CLV Segments Summary ===')
clv_df.groupby('clv_segment').agg({
    'User_Name': 'count',
    'total_spend': 'mean',
    'transaction_count': 'mean',
    'avg_order_value': 'mean',
    'simple_clv': 'mean'
}).round(2)

## 10. Category Affinity by Demographics

In [ ]:
# Create age groups
df['Age_Group'] = pd.cut(df['Age'], bins=[17, 25, 35, 45, 55, 70], labels=['18-25', '26-35', '36-45', '46-55', '56-70'])

# Product preferences by age group
age_category = pd.crosstab(df['Age_Group'], df['Product_Category'], normalize='index') * 100

print('=== Product Category Preferences by Age Group (%) ===')
print(age_category.round(2))

# Heatmap of age group vs category
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(age_category, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Percentage %'})
ax.set_title('Product Category Preferences by Age Group (%)', fontsize=14)
ax.set_xlabel('Product Category')
ax.set_ylabel('Age Group')
plt.tight_layout()
plt.show()

In [ ]:
# Category preferences by country
country_category = pd.crosstab(df['Country'], df['Product_Category'], normalize='index') * 100

print('=== Product Category Preferences by Country (%) ===')
print(country_category.round(2))

# Heatmap of country vs category
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(country_category, annot=True, fmt='.1f', cmap='Blues', ax=ax, cbar_kws={'label': 'Percentage %'})
ax.set_title('Product Category Preferences by Country (%)', fontsize=14)
ax.set_xlabel('Product Category')
ax.set_ylabel('Country')
plt.tight_layout()
plt.show()

# Average spend by age group and category
age_category_spend = df.groupby(['Age_Group', 'Product_Category'])['Purchase_Amount'].mean().unstack()
print('\n=== Average Spend by Age Group and Category ($) ===')
print(age_category_spend.round(2))

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(age_category_spend, annot=True, fmt='.0f', cmap='Greens', ax=ax, cbar_kws={'label': 'Avg Spend ($)'})
ax.set_title('Average Spend by Age Group and Category ($)', fontsize=14)
ax.set_xlabel('Product Category')
ax.set_ylabel('Age Group')
plt.tight_layout()
plt.show()

## 11. Geographic Patterns

In [ ]:
# Revenue by country
country_revenue = df.groupby('Country').agg(
    total_revenue=('Purchase_Amount', 'sum'),
    transaction_count=('Transaction_ID', 'count'),
    avg_transaction=('Purchase_Amount', 'mean'),
    unique_customers=('User_Name', 'nunique')
).sort_values('total_revenue', ascending=False).reset_index()

print('=== Revenue by Country ===')
print(country_revenue.round(2))

# Visualize revenue by country
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total Revenue by Country
country_revenue_sorted = country_revenue.sort_values('total_revenue', ascending=True)
axes[0].barh(country_revenue_sorted['Country'], country_revenue_sorted['total_revenue'], color=sns.color_palette('viridis', len(country_revenue)))
axes[0].set_title('Total Revenue by Country')
axes[0].set_xlabel('Total Revenue ($)')
for i, v in enumerate(country_revenue_sorted['total_revenue']):
    axes[0].text(v + 10000, i, f'${v:,.0f}', va='center', fontsize=9)

# Average Transaction by Country
country_revenue_sorted2 = country_revenue.sort_values('avg_transaction', ascending=True)
axes[1].barh(country_revenue_sorted2['Country'], country_revenue_sorted2['avg_transaction'], color=sns.color_palette('plasma', len(country_revenue)))
axes[1].set_title('Average Transaction Value by Country')
axes[1].set_xlabel('Avg Transaction ($)')
for i, v in enumerate(country_revenue_sorted2['avg_transaction']):
    axes[1].text(v + 2, i, f'${v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Category mix by country (revenue-based)
country_category_revenue = df.groupby(['Country', 'Product_Category'])['Purchase_Amount'].sum().unstack(fill_value=0)
country_category_pct = country_category_revenue.div(country_category_revenue.sum(axis=1), axis=0) * 100

print('=== Category Revenue Mix by Country (%) ===')
print(country_category_pct.round(2))

# Stacked bar chart of category mix by country
fig, ax = plt.subplots(figsize=(14, 8))
country_category_pct.plot(kind='bar', stacked=True, ax=ax, colormap='Set3')
ax.set_title('Category Revenue Mix by Country (%)')
ax.set_xlabel('Country')
ax.set_ylabel('Percentage of Revenue')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Payment method preferences by country
country_payment = pd.crosstab(df['Country'], df['Payment_Method'], normalize='index') * 100

print('\n=== Payment Method Preferences by Country (%) ===')
print(country_payment.round(2))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(country_payment, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Percentage %'})
ax.set_title('Payment Method Preferences by Country (%)', fontsize=14)
ax.set_xlabel('Payment Method')
ax.set_ylabel('Country')
plt.tight_layout()
plt.show()

## 12. Payment Method Analysis

In [ ]:
# Payment method distribution
payment_stats = df.groupby('Payment_Method').agg(
    transaction_count=('Transaction_ID', 'count'),
    total_revenue=('Purchase_Amount', 'sum'),
    avg_transaction=('Purchase_Amount', 'mean'),
    median_transaction=('Purchase_Amount', 'median')
).sort_values('total_revenue', ascending=False).reset_index()

payment_stats['pct_transactions'] = (payment_stats['transaction_count'] / payment_stats['transaction_count'].sum() * 100).round(2)
payment_stats['pct_revenue'] = (payment_stats['total_revenue'] / payment_stats['total_revenue'].sum() * 100).round(2)

print('=== Payment Method Statistics ===')
print(payment_stats.round(2))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Distribution of payment methods (pie chart)
axes[0].pie(payment_stats['transaction_count'], labels=payment_stats['Payment_Method'], autopct='%1.1f%%', 
            colors=sns.color_palette('Set2', len(payment_stats)), startangle=90)
axes[0].set_title('Payment Method Distribution')

# Average transaction by payment method
payment_sorted = payment_stats.sort_values('avg_transaction', ascending=True)
bars = axes[1].barh(payment_sorted['Payment_Method'], payment_sorted['avg_transaction'], 
                    color=sns.color_palette('coolwarm', len(payment_stats)))
axes[1].set_title('Average Transaction Value by Payment Method')
axes[1].set_xlabel('Avg Transaction ($)')
for i, v in enumerate(payment_sorted['avg_transaction']):
    axes[1].text(v + 2, i, f'${v:.2f}', va='center')

# Total revenue by payment method
payment_sorted2 = payment_stats.sort_values('total_revenue', ascending=True)
axes[2].barh(payment_sorted2['Payment_Method'], payment_sorted2['total_revenue'], 
             color=sns.color_palette('viridis', len(payment_stats)))
axes[2].set_title('Total Revenue by Payment Method')
axes[2].set_xlabel('Total Revenue ($)')
for i, v in enumerate(payment_sorted2['total_revenue']):
    axes[2].text(v + 10000, i, f'${v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Payment preferences by demographics
# By age group
age_payment = pd.crosstab(df['Age_Group'], df['Payment_Method'], normalize='index') * 100

print('=== Payment Method Preferences by Age Group (%) ===')
print(age_payment.round(2))

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(age_payment, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Percentage %'})
ax.set_title('Payment Method Preferences by Age Group (%)', fontsize=14)
ax.set_xlabel('Payment Method')
ax.set_ylabel('Age Group')
plt.tight_layout()
plt.show()

# Payment method by category
category_payment = pd.crosstab(df['Product_Category'], df['Payment_Method'], normalize='index') * 100

print('\n=== Payment Method Preferences by Product Category (%) ===')
print(category_payment.round(2))

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(category_payment, annot=True, fmt='.1f', cmap='PuRd', ax=ax, cbar_kws={'label': 'Percentage %'})
ax.set_title('Payment Method Preferences by Product Category (%)', fontsize=14)
ax.set_xlabel('Payment Method')
ax.set_ylabel('Product Category')
plt.tight_layout()
plt.show()

## 13. Repeat Purchase Analysis

In [ ]:
# Time between purchases per customer
df_sorted = df.sort_values(['User_Name', 'Transaction_Date'])
df_sorted['prev_purchase'] = df_sorted.groupby('User_Name')['Transaction_Date'].shift(1)
df_sorted['days_between'] = (df_sorted['Transaction_Date'] - df_sorted['prev_purchase']).dt.days

# Calculate average time between purchases per customer
customer_frequency = df_sorted.groupby('User_Name')['days_between'].agg(['mean', 'median', 'count']).reset_index()
customer_frequency.columns = ['User_Name', 'avg_days_between', 'median_days_between', 'repeat_purchases']

print('=== Time Between Purchases Summary ===')
print(f"Average days between purchases: {customer_frequency['avg_days_between'].mean():.1f}")
print(f"Median days between purchases: {customer_frequency['median_days_between'].median():.1f}")

# Distribution of days between purchases
valid_days = df_sorted['days_between'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(valid_days[valid_days <= 30], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Days Between Purchases (0-30 days)')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Frequency')
axes[0].axvline(valid_days.mean(), color='red', linestyle='--', label=f'Mean: {valid_days.mean():.1f} days')
axes[0].legend()

axes[1].hist(valid_days, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_title('Days Between Purchases (All)')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Frequency')
axes[1].axvline(valid_days.median(), color='orange', linestyle='--', label=f'Median: {valid_days.median():.1f} days')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\n=== Days Between Purchases Statistics ===')
print(valid_days.describe())

In [ ]:
# Repeat purchase rate by category
# Calculate how often customers repurchase in the same category
customer_category_counts = df.groupby(['User_Name', 'Product_Category']).size().reset_index(name='purchase_count')
repeat_by_category = customer_category_counts[customer_category_counts['purchase_count'] > 1].groupby('Product_Category').agg(
    customers_with_repeat=('User_Name', 'count'),
    avg_repeat_purchases=('purchase_count', 'mean')
).reset_index()

total_customers_by_category = customer_category_counts.groupby('Product_Category')['User_Name'].count().reset_index(name='total_customers')
repeat_by_category = repeat_by_category.merge(total_customers_by_category, on='Product_Category')
repeat_by_category['repeat_rate'] = (repeat_by_category['customers_with_repeat'] / repeat_by_category['total_customers'] * 100).round(2)

print('=== Repeat Purchase Rate by Category ===')
print(repeat_by_category.sort_values('repeat_rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Repeat rate by category
repeat_sorted = repeat_by_category.sort_values('repeat_rate', ascending=True)
axes[0].barh(repeat_sorted['Product_Category'], repeat_sorted['repeat_rate'], color=sns.color_palette('viridis', len(repeat_sorted)))
axes[0].set_title('Repeat Purchase Rate by Category (%)')
axes[0].set_xlabel('Repeat Rate (%)')
for i, v in enumerate(repeat_sorted['repeat_rate']):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center')

# Average repeat purchases by category
repeat_sorted2 = repeat_by_category.sort_values('avg_repeat_purchases', ascending=True)
axes[1].barh(repeat_sorted2['Product_Category'], repeat_sorted2['avg_repeat_purchases'], color=sns.color_palette('plasma', len(repeat_sorted)))
axes[1].set_title('Average Repeat Purchases by Category')
axes[1].set_xlabel('Avg Repeat Purchases')
for i, v in enumerate(repeat_sorted2['avg_repeat_purchases']):
    axes[1].text(v + 0.1, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Customer retention over time (monthly active customers)
df['month'] = df['Transaction_Date'].dt.to_period('M')
monthly_customers = df.groupby('month')['User_Name'].nunique().reset_index()
monthly_customers.columns = ['month', 'active_customers']
monthly_customers['month_str'] = monthly_customers['month'].astype(str)

# Calculate rolling retention
total_customers = df['User_Name'].nunique()
monthly_customers['retention_rate'] = (monthly_customers['active_customers'] / total_customers * 100).round(2)

print('=== Monthly Active Customers ===')
print(f"Total unique customers: {total_customers}")
print(f"Average monthly active customers: {monthly_customers['active_customers'].mean():.0f}")
print(f"Average monthly retention rate: {monthly_customers['retention_rate'].mean():.1f}%")

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Monthly active customers
axes[0].plot(range(len(monthly_customers)), monthly_customers['active_customers'], marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Monthly Active Customers Over Time')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Active Customers')
axes[0].set_xticks(range(0, len(monthly_customers), 3))
axes[0].set_xticklabels(monthly_customers['month_str'].iloc[::3], rotation=45)
axes[0].axhline(monthly_customers['active_customers'].mean(), color='red', linestyle='--', label=f"Avg: {monthly_customers['active_customers'].mean():.0f}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Monthly retention rate
axes[1].fill_between(range(len(monthly_customers)), monthly_customers['retention_rate'], alpha=0.3, color='green')
axes[1].plot(range(len(monthly_customers)), monthly_customers['retention_rate'], marker='o', color='green', linewidth=2)
axes[1].set_title('Monthly Retention Rate Over Time')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Retention Rate (%)')
axes[1].set_xticks(range(0, len(monthly_customers), 3))
axes[1].set_xticklabels(monthly_customers['month_str'].iloc[::3], rotation=45)
axes[1].axhline(monthly_customers['retention_rate'].mean(), color='red', linestyle='--', label=f"Avg: {monthly_customers['retention_rate'].mean():.1f}%")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Cohort Analysis

In [ ]:
# Group customers by first purchase month (cohort)
customer_first_purchase = df.groupby('User_Name')['Transaction_Date'].min().reset_index()
customer_first_purchase.columns = ['User_Name', 'first_purchase_date']
customer_first_purchase['cohort'] = customer_first_purchase['first_purchase_date'].dt.to_period('M')

# Merge cohort info back to transactions
df_cohort = df.merge(customer_first_purchase[['User_Name', 'cohort']], on='User_Name')
df_cohort['transaction_month'] = df_cohort['Transaction_Date'].dt.to_period('M')

# Calculate cohort age (months since first purchase)
df_cohort['cohort_age'] = (df_cohort['transaction_month'].astype('int64') - df_cohort['cohort'].astype('int64'))

print('=== Cohort Summary ===')
cohort_summary = customer_first_purchase.groupby('cohort').size().reset_index(name='customers')
print(f"Number of cohorts: {cohort_summary['cohort'].nunique()}")
print(f"Average cohort size: {cohort_summary['customers'].mean():.1f}")
print(f"\nFirst 12 cohorts:")
print(cohort_summary.head(12))

In [ ]:
# Cohort retention analysis - track unique customers over time
cohort_retention = df_cohort.groupby(['cohort', 'cohort_age'])['User_Name'].nunique().reset_index()
cohort_retention.columns = ['cohort', 'cohort_age', 'customers']

# Get initial cohort sizes
cohort_sizes = cohort_retention[cohort_retention['cohort_age'] == 0][['cohort', 'customers']].rename(columns={'customers': 'initial_size'})
cohort_retention = cohort_retention.merge(cohort_sizes, on='cohort')
cohort_retention['retention_rate'] = (cohort_retention['customers'] / cohort_retention['initial_size'] * 100).round(2)

# Pivot for heatmap
retention_pivot = cohort_retention.pivot(index='cohort', columns='cohort_age', values='retention_rate')

# Limit to first 12 months for readability
retention_pivot_display = retention_pivot.iloc[:, :12]

print('=== Cohort Retention Rates (%) ===')
print(retention_pivot_display.round(1))

# Retention heatmap
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(retention_pivot_display, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Retention Rate %'})
ax.set_title('Cohort Retention Analysis (% of Initial Customers)', fontsize=14)
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
plt.tight_layout()
plt.show()

In [ ]:
# Cohort spending analysis - track revenue over time
cohort_spending = df_cohort.groupby(['cohort', 'cohort_age'])['Purchase_Amount'].sum().reset_index()
cohort_spending.columns = ['cohort', 'cohort_age', 'total_spend']

# Pivot for heatmap
spending_pivot = cohort_spending.pivot(index='cohort', columns='cohort_age', values='total_spend')
spending_pivot_display = spending_pivot.iloc[:, :12]

print('=== Cohort Cumulative Spending ($) ===')

# Spending heatmap
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(spending_pivot_display, annot=True, fmt=',.0f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Total Spend ($)'})
ax.set_title('Cohort Spending Over Time ($)', fontsize=14)
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
plt.tight_layout()
plt.show()

# Average spend per customer per cohort age
cohort_avg_spend = df_cohort.groupby(['cohort', 'cohort_age']).agg(
    total_spend=('Purchase_Amount', 'sum'),
    customers=('User_Name', 'nunique')
).reset_index()
cohort_avg_spend['avg_spend_per_customer'] = cohort_avg_spend['total_spend'] / cohort_avg_spend['customers']

avg_spend_pivot = cohort_avg_spend.pivot(index='cohort', columns='cohort_age', values='avg_spend_per_customer')
avg_spend_pivot_display = avg_spend_pivot.iloc[:, :12]

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(avg_spend_pivot_display, annot=True, fmt='.0f', cmap='PuBu', ax=ax, cbar_kws={'label': 'Avg Spend per Customer ($)'})
ax.set_title('Average Spend per Customer by Cohort ($)', fontsize=14)
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
plt.tight_layout()
plt.show()

In [ ]:
# Identify best-performing cohorts
cohort_performance = df_cohort.groupby('cohort').agg(
    total_customers=('User_Name', 'nunique'),
    total_transactions=('Transaction_ID', 'count'),
    total_revenue=('Purchase_Amount', 'sum'),
    avg_transaction=('Purchase_Amount', 'mean')
).reset_index()

cohort_performance['revenue_per_customer'] = cohort_performance['total_revenue'] / cohort_performance['total_customers']
cohort_performance['transactions_per_customer'] = cohort_performance['total_transactions'] / cohort_performance['total_customers']

print('=== Best Performing Cohorts by Revenue per Customer ===')
top_cohorts = cohort_performance.nlargest(10, 'revenue_per_customer')
print(top_cohorts[['cohort', 'total_customers', 'total_revenue', 'revenue_per_customer', 'transactions_per_customer']].round(2))

# Visualize cohort performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Revenue per customer by cohort
axes[0, 0].bar(range(len(cohort_performance)), cohort_performance['revenue_per_customer'], color='steelblue', alpha=0.7)
axes[0, 0].set_title('Revenue per Customer by Cohort')
axes[0, 0].set_xlabel('Cohort Index')
axes[0, 0].set_ylabel('Revenue per Customer ($)')
axes[0, 0].axhline(cohort_performance['revenue_per_customer'].mean(), color='red', linestyle='--', label=f"Avg: ${cohort_performance['revenue_per_customer'].mean():,.0f}")
axes[0, 0].legend()

# Transactions per customer by cohort
axes[0, 1].bar(range(len(cohort_performance)), cohort_performance['transactions_per_customer'], color='coral', alpha=0.7)
axes[0, 1].set_title('Transactions per Customer by Cohort')
axes[0, 1].set_xlabel('Cohort Index')
axes[0, 1].set_ylabel('Transactions per Customer')
axes[0, 1].axhline(cohort_performance['transactions_per_customer'].mean(), color='red', linestyle='--', label=f"Avg: {cohort_performance['transactions_per_customer'].mean():.1f}")
axes[0, 1].legend()

# Total revenue by cohort
axes[1, 0].bar(range(len(cohort_performance)), cohort_performance['total_revenue'], color='green', alpha=0.7)
axes[1, 0].set_title('Total Revenue by Cohort')
axes[1, 0].set_xlabel('Cohort Index')
axes[1, 0].set_ylabel('Total Revenue ($)')

# Average transaction value by cohort
axes[1, 1].bar(range(len(cohort_performance)), cohort_performance['avg_transaction'], color='purple', alpha=0.7)
axes[1, 1].set_title('Average Transaction Value by Cohort')
axes[1, 1].set_xlabel('Cohort Index')
axes[1, 1].set_ylabel('Avg Transaction ($)')
axes[1, 1].axhline(cohort_performance['avg_transaction'].mean(), color='red', linestyle='--', label=f"Avg: ${cohort_performance['avg_transaction'].mean():.2f}")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print('\n=== Cohort Performance Summary ===')
print(f"Best cohort by revenue/customer: {cohort_performance.loc[cohort_performance['revenue_per_customer'].idxmax(), 'cohort']} (${cohort_performance['revenue_per_customer'].max():,.2f})")
print(f"Best cohort by transactions/customer: {cohort_performance.loc[cohort_performance['transactions_per_customer'].idxmax(), 'cohort']} ({cohort_performance['transactions_per_customer'].max():.1f})")
print(f"Highest revenue cohort: {cohort_performance.loc[cohort_performance['total_revenue'].idxmax(), 'cohort']} (${cohort_performance['total_revenue'].max():,.2f})")

## 15. Purchase Pattern Summary & Business Insights

### Key Findings

**Customer Lifetime Value:**
- CLV analysis enables segmentation of customers into Low, Medium, High, and Premium tiers
- Premium customers drive disproportionate revenue and should be prioritized for retention

**Category Affinity:**
- Different age groups and countries show distinct category preferences
- Targeted marketing can leverage demographic-specific product affinities

**Geographic Patterns:**
- Revenue distribution varies by country with distinct payment method preferences
- Category mix differs geographically, enabling localized product strategies

**Payment Methods:**
- Payment preferences correlate with demographics and purchase categories
- Digital payment adoption varies by region

**Repeat Purchase Behavior:**
- Time between purchases provides insights for re-engagement timing
- Category-specific repeat rates indicate which products drive loyalty

**Cohort Performance:**
- Early cohorts typically show higher lifetime engagement
- Cohort analysis identifies acquisition periods that yielded the best customers